# Figure 5 — a stochastic model of pre-leukaemic clonal evolution

This notebook reproduces the simulated panels of **Figure 5**: the distribution of
evolutionary patterns (**5b**), the age-incidence of AML (**5c**), the variant-frequency
trajectories preceding AML (**5d**), and the joint variant-frequency distribution used to
detect early clonal sweeps (**5e**). It is self-contained: run the cells top to bottom.

## Model

Blood is maintained by $N$ haematopoietic stem cells that acquire driver mutations at a rate
$U$ per year. Each driver confers a fitness effect $s$ drawn from a stretched-exponential
distribution of fitness effects (DFE). The DFE for the $k$-th driver in a lineage has a scale
that is $r$-fold larger than that of the $(k\!-\!1)$-th driver:

$$\rho(s)\;\propto\;\exp\!\left[-\left(\frac{s-a}{\,r^{\,k-1}\,s_0\,}\right)^{p}\right],\qquad s\ge 0,$$

where $s_0$ is the single-mutant DFE scale, $p$ the stretch exponent, and $a$ an offset of the
distribution. The published model uses $a=0$, which recovers Eq (3) of the paper.

Clones grow or shrink deterministically according to their fitness relative to the population
mean, with Poisson sampling modelling the stochastic drift. A simulated AML is defined as a clone that acquires **four** driver
mutations and exceeds **50 % cell fraction** before age 85.

## Parameters

| symbol | variable | value | meaning |
|---|---|---|---|
| $N$ | `N` | $10^5$ | number of stem cells |
| $U$ | `U` | $10^{-5}$/yr | driver mutation rate |
| $s_0$ | `s0` | 0.16 | DFE scale of the single-mutant fitness effect |
| $r$ | `r` | 2.5 | fold-increase in DFE scale per additional driver |
| $p$ | `p` | 3 | DFE stretch exponent |
| $a$ | `a` | 0 | DFE offset ($a=0$ is Eq (3) of the paper) |

Runtime scales with `N_SIMS` (set in section 4). The default (40,000) runs in ~10 minutes; the
published panels used 200,200 simulations (5b, 5c, 5e) and 40,000 (5d), produced with the
parallel scripts listed in section 9.

## 1. Imports and colours

In [ ]:
import numpy as np
import random
import scipy.integrate as integrate
from scipy.interpolate import make_smoothing_spline
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
%matplotlib inline

GREY = (0.55, 0.55, 0.55)     # observed data
PINK = (1.00, 0.18, 0.33)     # simulated
# clone colour by number of driver mutations (1-4)
MUT_COLORS = {1: '#4292c6', 2: '#41ab5d', 3: '#fd8d3c', 4: '#cb181d'}

## 2. Distribution of fitness effects

Fitness effects are sampled from the stretched-exponential DFE
$\exp\!\left[-((s-a)/(r^{k-1}s_0))^{p}\right]$, where $a$ is the DFE offset (`a` in section 4;
the published model uses $a=0$, Eq (3)). Each DFE is discretised once into a lookup table for
fast inverse-transform sampling.

In [ ]:
# normalised cumulative of the DFE  exp(-(( s - a )/sb)**p)  on the support [0, 100*sb].
# a is the DFE offset; a = 0 gives paper eq (3).
def dfe_cumulative(target_s, p, sb, a=0.0):
    density = lambda s: np.exp(-((s - a) / sb) ** p)
    s_lim = 100 * sb
    norm = integrate.quad(density, 0.0, s_lim)[0]
    return integrate.quad(density, 0.0, target_s)[0] / norm

# build a 1000-quantile lookup table mapping a uniform draw -> fitness effect
def build_dfe_sampler(p, sb, a=0.0, n_quantiles=1000):
    table, prev, s_lim = {}, 0, 100 * sb
    for target_s in np.linspace(0, s_lim, 1001):
        q = int(np.round(dfe_cumulative(target_s, p, sb, a) * n_quantiles))
        for k in range(prev, q):
            table[k] = target_s
        prev = q
        if q == n_quantiles:
            table[q] = target_s
            break
    return table

# draw one fitness effect from a DFE lookup table
def sample_fitness(table):
    return table[int(np.round(random.random() * (len(table) - 1)))]

## 3. Evolutionary dynamics

The population is a dictionary of clones. Each clone stores its current size, its full
size-vs-time trajectory, the set of driver mutations it carries (with their fitness effects),
and its net fitness (the sum of those effects). Each timestep: (i) clones acquire new drivers,
(ii) clones are resized by their fitness relative to the mean, (iii) extinct clones are pruned.

In [ ]:
# (i) acquire new driver mutations this timestep (Poisson), up to 4 drivers per clone
def mutate_population(clones, counter, U, dfes, t, dt, T):
    m = counter['n']
    # iterate a snapshot so clones born this step are not themselves mutated until next step
    snapshot = [(cid, e['current_clone_size'], e['mutations']) for cid, e in clones.items()]
    for cid, size, muts in snapshot:
        k = len(muts)
        if k >= 4 or size <= 0:
            continue
        for _ in range(np.random.poisson(size * U * dt)):
            m += 1
            child_muts = dict(muts)
            child_muts[m] = sample_fitness(dfes[k])          # k-th driver, scale r^k * s0
            traj = np.zeros(T); traj[t] = 1
            clones[m] = {'clone_size_trajectory': traj, 'current_clone_size': 1,
                         'mutations': child_muts, 'fitness': sum(child_muts.values()),
                         'children': [], 'aml_clone': False}
            clones[cid]['children'].append(m)
    counter['n'] = m

# population mean fitness (frequency-weighted)
def mean_fitness(clones):
    tot = sum(e['current_clone_size'] for e in clones.values())
    return sum(e['fitness'] * e['current_clone_size'] for e in clones.values()) / tot

# (ii) grow/shrink each clone by exp((fitness - mean_fitness) * dt), with Poisson sampling
def select_population(clones, mbar, t, dt):
    for e in clones.values():
        expected = e['current_clone_size'] * np.exp((e['fitness'] - mbar) * dt)
        e['current_clone_size'] = np.random.poisson(expected) if expected > 0 else 0
        e['clone_size_trajectory'][t] = e['current_clone_size']

# AML once any 4-driver clone exceeds 50% cell fraction
def aml_diagnosis(clones):
    tot = sum(e['current_clone_size'] for e in clones.values())
    dx = False
    for e in clones.values():
        if len(e['mutations']) == 4 and e['current_clone_size'] / tot > 0.5:
            e['aml_clone'] = True
            dx = True
    return dx

# (iii) drop extinct clones that never reached size 10 and have no descendants (memory only)
def purge(clones):
    dead = [c for c, e in clones.items()
            if e['current_clone_size'] == 0 and not e['children']
            and e['clone_size_trajectory'].max() < 10]
    for c in dead:
        del clones[c]

def population_size_trajectory(clones, T):
    tot = np.zeros(T)
    for e in clones.values():
        tot += e['clone_size_trajectory']
    return tot

# cell-fraction trajectory of every mutation (summed over all clones carrying it);
# n_drivers records the fewest drivers of any clone carrying the mutation (i.e. its order)
def variant_trajectories(clones, T):
    pop = population_size_trajectory(clones, T)
    traj = {}
    for e in clones.values():
        k = len(e['mutations'])
        # 0 where the population is extinct (after diagnosis) to avoid divide-by-zero
        frac = np.divide(e['clone_size_trajectory'], pop, out=np.zeros(T), where=pop > 0)
        for mid in e['mutations']:
            if mid in traj:
                traj[mid]['cell_fraction'] += frac
                traj[mid]['n_drivers'] = min(traj[mid]['n_drivers'], k)
            else:
                traj[mid] = {'cell_fraction': frac.copy(), 'n_drivers': k}
    return traj

# largest and second-largest variant VAF at a time index (VAF = half the cell fraction);
# returns None if the individual is no longer alive (population extinct after AML)
def largest_two_vaf(clones, t_index):
    pop = sum(e['clone_size_trajectory'][t_index] for e in clones.values())
    if pop <= 0:
        return None
    sizes = {}
    for e in clones.values():
        s = e['clone_size_trajectory'][t_index]
        if s > 0:
            for mid in e['mutations']:
                sizes[mid] = sizes.get(mid, 0.0) + s
    if not sizes:
        return (0.0, 0.0)
    vafs = sorted((0.5 * v / pop for v in sizes.values()), reverse=True)
    return vafs[0], (vafs[1] if len(vafs) > 1 else 0.0)

## 4. Run the simulation

One pass produces everything the panels need: the future-AML cases (with their clonal
histories) for panels 5b and 5d, the diagnosis ages for panel 5c, and the largest/second
variant VAFs at ages 40/50/60/70 for every individual for panel 5e.

In [ ]:
# ---- model parameters ----
N   = 10**5      # stem cells
U   = 1e-5       # driver mutation rate per year
s0  = 0.16       # DFE scale, single-mutant (r-fold larger for each further driver)
r   = 2.5
p   = 3          # DFE stretch exponent
a   = 0.0        # DFE offset (0 = paper eq (3))
T, dt = 850, 0.1 # 850 steps of 0.1 yr -> age 85
N_SIMS = 40000   # ~10 min. The published panels used 200,000 sims (5b/5c/5e) and 40,000 (5d)
                 # via the parallel scripts (see section 9); raise N_SIMS for smoother panels.

AGES = [40, 50, 60, 70]                 # ages at which panel 5e samples the blood
age_idx = [int(age / dt) for age in AGES]

random.seed(1); np.random.seed(1)
dfes = [build_dfe_sampler(p, s0 * r**k, a) for k in range(4)]   # one DFE per driver number 0..3

simulated_cases = {}                    # sim -> {'clones', 'diagnosis_time'}  (future AML)
dx_ages = []                            # diagnosis ages                       (panel 5c)
case_vaf = []                           # per case: {age: (largest, 2nd) VAF or None}  (panel 5e)
ctrl_vaf = {age: [] for age in AGES}    # controls pooled per age

for sim in range(N_SIMS):
    root = {'clone_size_trajectory': np.zeros(T), 'current_clone_size': N,
            'mutations': {}, 'fitness': 0.0, 'children': [], 'aml_clone': False}
    root['clone_size_trajectory'][0] = N
    clones = {0: root}; counter = {'n': 0}; aml = False
    for t in range(T):
        mutate_population(clones, counter, U, dfes, t, dt, T)
        select_population(clones, mean_fitness(clones), t, dt)
        purge(clones)
        if aml_diagnosis(clones):
            aml = True
            break
    if aml:
        simulated_cases[sim] = {'clones': clones, 'diagnosis_time': t * dt}
        dx_ages.append(t * dt)
        case_vaf.append({age: largest_two_vaf(clones, ti) for age, ti in zip(AGES, age_idx)})
    else:
        for age, ti in zip(AGES, age_idx):
            v = largest_two_vaf(clones, ti)
            if v is not None:
                ctrl_vaf[age].append(v)

print(f"{N_SIMS} simulations -> {len(simulated_cases)} future AML cases "
      f"({100 * len(simulated_cases) / N_SIMS:.2f}%)")

## 5. Panel 5b — evolutionary pattern

Each virtual AML is classified as **late** (the first driver of the AML lineage only becomes
detectable — VAF > 0.1 % — within 2 years of diagnosis), **branched** (an off-trunk clone with
$\geq$1 driver ever exceeds 10 % cell fraction, i.e. clonal interference), or **linear**
(a single dominant trunk). Observed counts are the $n=47$ pre-AML cases.

In [ ]:
def classify_case(clones, dx_time, T, dt):
    pop = population_size_trajectory(clones, T) + 0.1        # +0.1 avoids /0 at diagnosis
    trunk = None
    for e in clones.values():
        if len(e['mutations']) == 4 and e['aml_clone']:
            trunk = list(e['mutations'])
            break
    first_driver, trunk = trunk[0], set(trunk)
    cf = np.zeros(T)                                          # cell fraction of the first driver
    for e in clones.values():
        if first_driver in e['mutations']:
            cf += e['clone_size_trajectory']
    if dx_time - np.argmax(cf / pop >= 0.002) * dt <= 2.0:    # 0.2% cell fraction = 0.1% VAF
        return 'late'
    for cid, e in clones.items():
        if len(e['mutations']) > 0 and cid not in trunk \
           and (e['clone_size_trajectory'] / pop).max() > 0.1:
            return 'branched'
    return 'linear'

pattern = [classify_case(v['clones'], v['diagnosis_time'], T, dt) for v in simulated_cases.values()]
n = len(pattern)
sim_counts = {k: pattern.count(k) for k in ['linear', 'branched', 'late']}    
obs_counts = {'linear': 26, 'branched': 11, 'late': 10}       # observed pre-AML cases (n=47)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ypos, h = np.arange(3)[::-1], 0.36
for i, c in enumerate(['linear', 'branched', 'late']):
    y = ypos[i]; op, sp = 100 * obs_counts[c] / 47, 100 * sim_counts[c] / n
    ax.barh(y + h/2 + 0.02, op, height=h, color=GREY)
    ax.barh(y - h/2 - 0.02, sp, height=h, color=PINK)
    ax.text(op + 1.5, y + h/2 + 0.02, f"{obs_counts[c]}/47", va='center', fontsize=9, color=GREY)
    ax.text(sp + 1.5, y - h/2 - 0.02, f"{sim_counts[c]}/{n}", va='center', fontsize=9, color=PINK)
ax.set_yticks(ypos); ax.set_yticklabels(['linear', 'branched', 'late'])
ax.set_xlim(0, 100); ax.set_xlabel('%'); ax.set_title('Figure 5b  evolutionary pattern')
ax.legend(handles=[Patch(color=GREY, label='observed (n=47)'),
                   Patch(color=PINK, label=f'simulated (n={n})')], loc='lower right', frameon=False)
ax.spines[['top', 'right']].set_visible(False); plt.tight_layout()

plt.show()
for c in ['linear', 'branched', 'late']:
    print(f"{c:9s}: simulated {100*sim_counts[c]/n:4.1f}%   observed {100*obs_counts[c]/47:4.1f}%")

## 6. Panel 5c — age-incidence of AML

The simulated incidence (per 100,000 per year, in 5-year bands) is compared with the observed
age-specific incidence of AML (Cancer Research UK). The pink line is a smoothing spline through
the binned simulated rates.

In [ ]:
# observed AML age-specific incidence, per 100,000 per year, 5-year bands (Cancer Research UK)
cruk_m = np.array([1.1,0.4,0.6,0.7,1.0,0.9,1.2,1.3,1.6,2.1,3.3,4.8,7.5,12.6,18.2,26.7,34.0])
cruk_f = np.array([0.9,0.4,0.5,0.7,0.8,1.0,1.3,1.2,1.7,2.1,2.8,3.5,5.5,7.9,10.9,13.6,20.4])
cruk = 0.5 * (cruk_m + cruk_f)
band_ages = np.linspace(2.5, 82.5, 17)

counts = np.histogram(dx_ages, np.linspace(0, 85, 18))[0]
incidence = 100000 * counts / (5 * N_SIMS)
cum_sim, cum_obs = incidence.sum() * 5 / 1e5, cruk.sum() * 5 / 1e5

fig, ax = plt.subplots(figsize=(6.5, 5))
m = band_ages >= 30
spline = make_smoothing_spline(band_ages[m], np.sqrt(incidence[m]), lam=8.0)   # variance-stabilised
xf = np.linspace(30, 82.5, 400); yf = np.clip(spline(xf), 0, None) ** 2
ax.plot(xf, yf, color=PINK, lw=3, label=f'simulated (lifetime risk {cum_sim*100:.2f}%)')
ax.scatter(band_ages, cruk, color=GREY, s=70, edgecolors='white',
           label=f'observed CRUK ({cum_obs*100:.2f}%)')
ax.set_xlim(30, 85); ax.set_ylim(0, max(45, yf.max() * 1.1))
ax.set_xlabel('age (years)'); ax.set_ylabel('incidence (per 100,000 / yr)')
ax.set_title('Figure 5c  age-incidence of AML')
ax.legend(loc='upper left', frameon=False); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Panel 5d — variant trajectories before AML

Variant-frequency trajectories in the 15 years before diagnosis, coloured by number of driver
mutations (blue 1, green 2, orange 3, red 4): a 50-case faint ensemble with a few **bold focal
trajectories** showing characteristic behaviours — single-mutant sweeps, a double-mutant up-down
reversal, a triple-mutant classic sweep and an up-down-up double reversal, and an emerging AML
clone — auto-detected from the simulated cases. Trajectories are sampled at 2000-4000x sequencing
depth. Rare reversal patterns appear only with a large case pool: increase `N_SIMS` (section 4),
or run `figure5d_pipeline.py`, for the full published panel.

In [ ]:
import matplotlib.patheffects as pe
from scipy.signal import find_peaks

YR5 = list(range(-15, 0))
COL   = {1: '#4292c6', 2: '#74c476', 3: '#feb24c', 4: '#ef3b2c'}          # colours of panel 5d
def _lighten(h, f=0.5):
    c = np.array([int(h[i:i+2], 16) for i in (1, 3, 5)])/255
    return '#%02x%02x%02x' % tuple(int(round(x*255)) for x in c*(1-f)+f)
COL_L = {k: _lighten(v) for k, v in COL.items()}

# annual variant trajectories (cell fraction %) in the 15 yr before diagnosis, per case
def case_trajectories(clones, dx_time):
    out = []
    for m in variant_trajectories(clones, T).values():
        ys, vals = [], []
        for yr in YR5:
            idx = int(round((dx_time + yr)/dt))
            if 0 <= idx < T:
                ys.append(yr); vals.append(m['cell_fraction'][idx]*100)
        if vals and max(vals) > 0.5:
            out.append({'yr': ys, 'cf': vals, 'k': m['n_drivers']})
    return out
cases_5d = [case_trajectories(v['clones'], v['diagnosis_time']) for v in simulated_cases.values()]

# behaviour detectors (run on the true annual trajectory; index 0 = -15 yr)
def _swept(y):  y=np.asarray(y,float); return y.min() if (y.min()>=82 and y.max()-y.min()<16) else None
def _finish(y):
    y=np.asarray(y,float)
    if y[-1]<95 or not (35<=y[0]<90): return None
    hi=np.where(y>=95)[0]
    return (100-hi[0])+(90-y[0])*0.2 if (len(hi) and hi[0]<=8 and y[hi[0]:].min()>=85) else None
def _updown(y):
    y=np.asarray(y,float); n=len(y); pi=int(np.argmax(y))
    if y.max()<15 or pi in (0,n-1): return None
    return y[pi]-y[pi:].min() if (y[pi]-y[pi:].min()>=12 and y[-1]<y[pi]-10 and y[:pi+1].min()<y[pi]-8) else None
def _osweep(y):
    y=np.asarray(y,float); n=len(y); pi=int(np.argmax(y))
    return y[-1] if (y.max()>=55 and y[-1]>=0.82*y.max() and pi>=n-4 and y.max()-y[0]>=25) else None
def _oudu(y):
    y=np.asarray(y,float); n=len(y)
    if y.max()<25 or y[0]>=25: return None
    mx,_=find_peaks(y,prominence=5); mn,_=find_peaks(-y,prominence=5); best=None
    for i in mx:
        for j in mn:
            if j>i and y[j]>2:
                dip=y[i]-y[j]; km=j+int(np.argmax(y[j:])); kick=y[km]-y[j]
                if dip>=8 and kick>=12 and km>=n-4 and y[i]-y[0]>=15: best=max(best if best is not None else -1, kick+dip*0.2)
    return best
def _red(y):
    y=np.asarray(y,float); n=len(y)
    return y[-1] if (10<=y[-1]<=20 and y[:n-2].max()<2.5) else None
DET = [('b_old',1,_swept),('b_fin',1,_finish),('g_ud',2,_updown),('o_sw',3,_osweep),('o_udu',3,_oudu),('red',4,_red)]

best = {}
for variants in cases_5d:
    for v in variants:
        for key, kk, fn in DET:
            if v['k']==kk:
                s = fn(v['cf'])
                if s is not None and s > best.get(key,(-1e9,None))[0]: best[key]=(s,v)
focal = [best[k][1] for k,_,_ in DET if k in best]
print('highlighted behaviours:', [k for k,_,_ in DET if k in best])

rng5 = np.random.default_rng(7)
def _seq(cf):                                       # sample at 2000-4000x depth (VAF = cf/2)
    out=[]
    for f in cf:
        d=int(rng5.integers(2000,4001)); out.append(rng5.binomial(d, min(max(f/200,0),1))/d*200)
    return out

samp = [cases_5d[i] for i in rng5.choice(len(cases_5d), min(50, len(cases_5d)), replace=False)]
fig, ax = plt.subplots(figsize=(13, 5.2))
for variants in samp:                               # faint ensemble
    for v in variants:
        if max(v['cf']) < 1: continue
        ax.plot(v['yr'], _seq(v['cf']), color=COL_L[v['k']], lw=1.8, alpha=0.85, zorder=2,
                solid_capstyle='round', path_effects=[pe.Stroke(linewidth=3.0, foreground='white'), pe.Normal()])
for v in focal:                                     # bold focal trajectories
    ax.plot(v['yr'], _seq(v['cf']), color=COL[v['k']], lw=4.5, zorder=6, solid_capstyle='round',
            path_effects=[pe.Stroke(linewidth=8.0, foreground='white'), pe.Normal()])
ax.set_xlim(-15, -1); ax.set_ylim(0, 105); ax.set_xticks(range(-15, 0, 2)); ax.set_yticks(range(0, 101, 20))
ax.set_xlabel('years before AML diagnosis'); ax.set_ylabel('fraction of cells (%)')
ax.legend(handles=[Patch(color=COL[k], label=f"{k} driver{'s' if k > 1 else ''}") for k in [1, 2, 3, 4]],
          loc='lower center', bbox_to_anchor=(0.5, 1.005), frameon=False, ncol=4, handlelength=1.4, columnspacing=1.6)
ax.spines[['top', 'right']].set_visible(False); plt.tight_layout(); plt.show()

## 8. Panel 5e — joint variant-frequency distribution and somatic-sweep risk

For a sample of 100 future-AML cases (pink) and the controls (grey) we plot the largest versus
second-largest variant VAF at each age; **n** in each panel is the number of sampled cases shown
(fewer at older ages, as some have already been diagnosed). Points right of the line
(largest VAF > 30 %) carry a **somatic sweep**. The odds ratio — computed from *all* cases and
controls, not just the plotted sample — quantifies how much a sweep raises the odds of future
AML; the age-adjusted (Mantel–Haenszel) value combines the four ages.

In [ ]:
# odds ratio of future AML given a somatic sweep (largest VAF > threshold)
def odds_ratio(cases_largest, controls_largest, threshold=0.30):
    A = sum(l > threshold for l in cases_largest);    B = len(cases_largest) - A
    C = sum(l > threshold for l in controls_largest); D = len(controls_largest) - C
    a_, b, c, d = (A, B, C, D) if min(A, B, C, D) else (A+.5, B+.5, C+.5, D+.5)
    return (a_ * d) / (b * c), A, B, C, D

rng_e = np.random.default_rng(0)
# plot a sample of 100 future-AML cases (the odds ratio still uses ALL cases and controls)
sample = [case_vaf[i] for i in rng_e.choice(len(case_vaf), min(100, len(case_vaf)), replace=False)]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
mh_num = mh_den = 0.0
for ax, age in zip(axes, AGES):
    orr, A, B, C, D = odds_ratio([rec[age][0] for rec in case_vaf if rec[age] is not None],
                                 [l for l, _ in ctrl_vaf[age]])
    mh_num += A * D / (A + B + C + D); mh_den += B * C / (A + B + C + D)
    cpts = [rec[age] for rec in sample if rec[age] is not None and rec[age][0] > 0]      # sampled cases at this age
    kpts = [(l, s) for l, s in ctrl_vaf[age] if l > 0]
    if len(kpts) > 4000:
        kpts = [kpts[i] for i in rng_e.choice(len(kpts), 4000, replace=False)]     # subsample controls to render
    ax.scatter([l for l, s in kpts], [s for l, s in kpts], color=GREY, s=30, alpha=0.3, linewidths=0)
    ax.scatter([l for l, s in cpts], [s for l, s in cpts], color=PINK, s=70, edgecolors='w', linewidths=0.7)
    ax.plot([0.3, 0.3], [-0.02, 0.52], color='k', lw=1)
    ax.set_xlim(-0.02, 0.52); ax.set_ylim(-0.02, 0.52); ax.set_aspect('equal')
    ax.set_xlabel('largest VAF'); ax.set_title(f'{age} years')
    if age == AGES[0]:
        ax.set_ylabel('second largest VAF')
    ax.text(0.31, 0.47, f'OR = {orr:.0f}', fontsize=11, fontweight='bold')
    ax.text(0.31, 0.40, f'n = {len(cpts)}', fontsize=10, color=PINK)               # cases shown at this age
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle(f'Figure 5e  joint VAF distribution   (sample of {len(sample)} cases; age-adjusted OR = {mh_num/mh_den:.0f})', y=1.03)
plt.tight_layout(); plt.show()

## 9. Reproducing the published figures (parallel scripts)

This notebook runs the whole model in one process at a modest `N_SIMS`. The published panels were
produced at 200,200 simulations (5b, 5c, 5e) and 40,000 (5d) with parallel scripts that call
`sim_fast.py`, a speed-optimised implementation of the identical model. All default to the
published parameters ($s_0=0.16$, $r=2.5$, $p=3$, offset $a=0$).

| script | reproduces | example command |
|---|---|---|
| `sim_fast.py` | the model itself (imported by every script below) | — |
| `figure5_pipeline.py` | **5b** evolutionary pattern + **5c** age-incidence | `python figure5_pipeline.py --sims 200000 --workers 4 --out Figure5_bc` |
| `fig5e_pipeline.py` | **5e** joint-VAF scatter + somatic-sweep odds ratios | `python fig5e_pipeline.py --sims 200000 --workers 4 --out Figure5e` |
| `figure5d_pipeline.py` | **5d** variant trajectories, 15 yr before AML | `python figure5d_pipeline.py --sims 40000 --workers 4 --out Figure5d_trajectories` |
| `figure5d_50yr_pipeline.py` + `run50yr.py` | **5d extended to 50 yr** before AML (supplementary) | `python run50yr.py 6500` &nbsp;(repeat until it prints `ALL DONE`) |
| `figure5d_controls.py` | **5d control panel** — 50 virtual controls | `python figure5d_controls.py --sims 8000 --out Figure5d_controls` |
| `figure5d_controls_x4.py` | **four replicate control panels** (supplementary) | `python figure5d_controls_x4.py --sims 2400` |

Each script writes a `.pdf` and `.png` (the 5b/5c and 5e scripts also write a `_summary.json` /
`_odds_ratios.json`). Pass `--s`, `--r`, `--p`, `--offset` to vary parameters and `--seed` to fix
the random stream.